# Prepare forest cover data

Predict seven forest cover types from terrain, wilderness area, and soil. This notebook profiles the public UCI Covertype data and writes a reproducible, stratified sample for the model comparison.


In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.datasets import fetch_covtype
from sklearn.model_selection import train_test_split

SEED = 0
SAMPLE_ROWS = 30_000
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

source = fetch_covtype(data_home=DATA_DIR / "covertype_cache", as_frame=True)
features = source.data.copy()
target = source.target.rename("Forest_Cover")
print(f"Source shape: {features.shape[0]:,} rows, {features.shape[1]} predictors")
display(target.value_counts().sort_index().rename("source_count"))


## Validate and collapse indicator columns

Each wilderness and soil indicator group should identify one category per row. Use an explicit unknown category if a group does not identify exactly one. Keep terrain fields, including valid negative vertical distances.


In [ ]:
for prefix, name in [("Wilderness_Area", "Area"), ("Soil_Type", "Soil_Type")]:
    indicators = features.filter(regex=f"^{prefix}").fillna(0).eq(1)
    features[name] = indicators.idxmax(axis=1).where(indicators.sum(axis=1).eq(1), "Unknown")
    features = features.drop(columns=indicators.columns)

if SAMPLE_ROWS is not None and SAMPLE_ROWS < len(features):
    features, _, target, _ = train_test_split(
        features, target, train_size=SAMPLE_ROWS, stratify=target, random_state=SEED,
    )

prepared = features.join(target)
prepared.to_csv(DATA_DIR / "forest_cover_prepared.csv", index=False)
display(prepared["Forest_Cover"].value_counts().sort_index().rename("prepared_count"))
print(f"Prepared rows: {len(prepared):,}")
